In [ ]:
#检查w3 attention的api
import inspect
from pathlib import Path
import sys

root = Path(r"D:\Projects\speech-ai-learning")
if root not in sys.path:
    sys.path.insert(0, str(root))

import src.attention as attention_module

print("===== W3 Attention API =====")

for name, obj in vars(attention_module).items():
    if getattr(obj, "__module__", None) != attention_module.__name__:
        continue

    if inspect.isclass(obj) or inspect.isfunction(obj):
        print(f"\n{name}: {inspect.signature(obj)}")

        if inspect.isclass(obj) and hasattr(obj, "forward"):
            print(
                "forward:",
                inspect.signature(obj.forward)
            )


===== W3 Attention API =====

dot_product_attention: (Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, scale: bool = True, valid_mask=None)

attention_entropy: (weights)

make_causal_mask: (T: int, device=None) -> torch.Tensor

make_padding_mask: (lengths: torch.Tensor, max_len: int) -> torch.Tensor

main: ()

test_mask: ()


In [ ]:
#检查w9d2的卷积模块
from src.conformer import ConformerConvModule

conv = ConformerConvModule(
    dim=64,
    kernel_size=31,
    dropout=0.1,
)

print(conv)

ConformerConvModule(
  (layernorm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (pointwise1): Conv1d(64, 128, kernel_size=(1,), stride=(1,))
  (glu): GLU(dim=1)
  (depthwise): Conv1d(64, 64, kernel_size=(31,), stride=(1,), padding=(15,), groups=64, bias=False)
  (batchnorm): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (silu): SiLU()
  (pointwise2): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
  (dropout): Dropout(p=0.1, inplace=False)
)


In [3]:
import inspect

from src.attention import (
    dot_product_attention,
    make_padding_mask,
)

print("===== dot_product_attention =====")
print(inspect.getsource(dot_product_attention))

print("===== make_padding_mask =====")
print(inspect.getsource(make_padding_mask))

===== dot_product_attention =====
def dot_product_attention(
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
        scale:bool = True,
        valid_mask=None
):
    #1.先做shape检查
    ## Q与K的特征维度必须相同
    assert Q.size(-1) == K.size(-1)

    ## KV序列长度必须相同
    assert K.size(-2) == V.size(-2)

    ## Batch-size一致
    assert Q.size(0) == K.size(0) == V.size(0)

    #2.计算scores
    scores = Q @ K.transpose(-2, -1)
    if scale:
        scores = scores / math.sqrt(Q.size(-1))

    #额外加入，如果输入valid_mask张量矩阵，那么进行mask操作
    if valid_mask is not None:
        assert valid_mask.dtype == torch.bool

        valid_mask = torch.broadcast_to(
            valid_mask,
            scores.shape,
        )

        masked_scores = scores.masked_fill(
            ~valid_mask,float("-inf")
        )

        valid_rows = valid_mask.any(
            dim=-1,keepdim=True
        )

        safe_scores = torch.where(
            valid_rows,
            masked_scores,
            torch.

In [4]:
import torch

lengths = torch.tensor([5, 3])

mask = make_padding_mask(
    lengths,
    max_len=5
)

print("Mask:")
print(mask)

print("Shape:", mask.shape)
print("Dtype:", mask.dtype)

Mask:
tensor([[ True,  True,  True,  True,  True],
        [ True,  True,  True, False, False]])
Shape: torch.Size([2, 5])
Dtype: torch.bool


In [5]:
import torch

B, H, T, D = 2, 4, 5, 8

q = torch.randn(B, H, T, D)
k = torch.randn(B, H, T, D)
v = torch.randn(B, H, T, D)

lengths = torch.tensor([5, 3])

valid_mask = make_padding_mask(lengths, T)

# 扩展 Mask
attn_mask = valid_mask[:, None, None, :]

output, weights = dot_product_attention(
    q, k, v,
    valid_mask=attn_mask
)

print("Output:", output.shape)
print("Weights:", weights.shape)

print("第二条语音 padded key 权重:")
print(weights[1, :, :, 3:])

Output: torch.Size([2, 4, 5, 8])
Weights: torch.Size([2, 4, 5, 5])
第二条语音 padded key 权重:
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]])


In [1]:
#测试完整ConformerBlock
import torch
from pathlib import Path
import sys

root = Path(r"D:\Projects\speech-ai-learning")
if root not in sys.path:
    sys.path.insert(0, str(root))

from src.conformer import ConformerBlock
from src.attention import make_padding_mask

model = ConformerBlock(
    dim=64,
    num_heads=4,
    ff_expansion=4,
    kernel_size=31,
    dropout=0.1
)

x = torch.randn(2,100,64,requires_grad=True)

lengths = torch.tensor([100,80])
valid_mask = make_padding_mask(lengths, 100)

#前向传播
y = model(x, valid_mask = valid_mask)

print("Input shape:", x.shape)
print("Output shape:", y.shape)

assert y.shape == x.shape

#反向传播
loss = y.square().mean()
loss.backward()

print("Input gradient exists:", x.grad is not None)

# 检查参数是否获得梯度
for name, param in model.named_parameters():
    if param.requires_grad and param.grad is None:
        print("Missing gradient:", name)

print("Contains NaN:", torch.isnan(y).any().item())

Input shape: torch.Size([2, 100, 64])
Output shape: torch.Size([2, 100, 64])
Input gradient exists: True
Contains NaN: False


### 实验前统一配置

In [2]:
import torch
from src.conformer import (
    ConformerMHSA,
    ConformerConvModule,
)
from src.attention import make_padding_mask

# 实验1使用：多头自注意力
mhsa = ConformerMHSA(
    dim=64,
    num_heads=4,
    dropout=0.1,
)

# 实验2使用：卷积模块
conv = ConformerConvModule(
    dim=64,
    kernel_size=31,
    dropout=0.1,
)

# 构造两条不同长度的语音特征
lengths = torch.tensor([100, 80])
valid_mask = make_padding_mask(lengths, max_len=100)

### 实验1：验证MHSA的Padding Mask

#### 目的：修改 Padding 位置的输入，观察有效位置的 MHSA 输出是否改变。

In [3]:
mhsa.eval()

x1 = torch.randn(2,100,64)
x2 = x1.clone()

# 修改第二条语音的 Padding 位置
x2[1, 80:] = torch.randn_like(x2[1, 80:]) * 10

with torch.no_grad():
    y1 = mhsa(x1, valid_mask)
    y2 = mhsa(x2, valid_mask)

#比较有效位置
diff = (y1[1, :80] - y2[1,:80]).abs().max()

print("Valid position max difference:", diff.item())

# 比较 Padding 位置
pad_diff = (y1[1, 80:] - y2[1, 80:]).abs().max()

print("Padding position max difference:", pad_diff.item())

Valid position max difference: 0.0
Padding position max difference: 0.10254096984863281


### 实验2：验证 Depthwise Conv 的 Padding 影响

#### 目的：观察 Padding 是否会影响附近的有效帧。

In [4]:
conv.eval()

x1 = torch.randn(2,100,64)
x1[1, 80:0] = 0

x2 = x1.clone()

#只修改padding部分
x2[1,80:] = torch.randn_like(x2[1, 80:]) * 10

with torch.no_grad():
    y1 = conv(x1)
    y2 = conv(x2)

# 远离 Padding 边界
far_diff = (
    y1[1, :65] - y2[1, :65]
).abs().max()

# 靠近 Padding 边界
boundary_diff = (
    y1[1, 65:80] - y2[1, 65:80]
).abs().max()

print("Far valid difference:", far_diff.item())
print("Boundary difference:", boundary_diff.item())

Far valid difference: 0.0
Boundary difference: 0.160967156291008


### 实验3：验证 BatchNorm 的 Padding 影响

#### 目的：观察训练模式下，改变 Padding 是否会影响远离边界的有效位置。

In [5]:
import copy

conv1 = ConformerConvModule(dim=64,kernel_size=31,dropout=0.0)

conv2 = copy.deepcopy(conv1)

conv1.train()
conv2.train()

x1 = torch.randn(2, 100, 64)
x1[1, 80:] = 0

x2 = x1.clone()
x2[1, 80:] = torch.randn_like(x2[1, 80:]) * 10

with torch.no_grad():
    y1 = conv1(x1)
    y2 = conv2(x2)

far_diff = (
    y1[1, :65] - y2[1, :65]
).abs().max()

print("Far valid difference:", far_diff.item())

Far valid difference: 0.0877302885055542
